# PY500 — Variations, Deviations, and Distributions

Understanding **how data varies** is just as important as knowing its central value. This notebook covers the fundamental measures of spread and the probability distributions that model real-world phenomena.

## Topics Covered
1. Measures of variation (variance, std, range, IQR, CV)
2. Normal (Gaussian) Distribution
3. Other important distributions (Binomial, Poisson, Uniform, Exponential)
4. Central Limit Theorem (CLT)

> **Perspective:** A mean without a measure of spread is like a weather forecast that says "average temperature is 25°C" — useless if the range is 5°C to 45°C. Always report variability alongside the average.

---
## 1. Measures of Variation

In [ ]:
## All measures of spread in one place

import numpy as np
import pandas as pd
from scipy import stats

data = np.array([45, 50, 55, 58, 60, 62, 65, 70, 75, 120])

print(f"Data     : {data}")
print(f"Mean     : {np.mean(data):.2f}")
print(f"Median   : {np.median(data):.2f}")
print(f"Range    : {np.ptp(data)}")
print(f"Variance (sample): {np.var(data, ddof=1):.2f}")
print(f"Std Dev  (sample): {np.std(data, ddof=1):.2f}")

q1, q3 = np.percentile(data, [25, 75])
iqr = q3 - q1
print(f"IQR      : {iqr:.2f} (Q1={q1}, Q3={q3})")

# Coefficient of Variation (CV) — relative measure of spread
cv = (np.std(data, ddof=1) / np.mean(data)) * 100
print(f"CV       : {cv:.1f}%")

# MAD — Median Absolute Deviation (robust)
mad = np.median(np.abs(data - np.median(data)))
print(f"MAD      : {mad:.2f}")

# Learning Point: CV allows comparing variability between datasets with
# different units or scales. A salary dataset with CV=30% is more "spread"
# than one with CV=10%, regardless of the actual salary values.

### Quick Reference: Measures of Spread

| Measure | Formula Intuition | Robust? | Use When |
|---|---|---|---|
| **Range** | max − min | No | Quick sense of spread |
| **Variance** | Avg of squared deviations from mean | No | Foundation for std |
| **Std Dev** | √variance | No | Same units as data |
| **IQR** | Q3 − Q1 | Yes | Data has outliers |
| **MAD** | Median of |x − median| | Yes | Robust alternative to std |
| **CV** | (std / mean) × 100% | No | Comparing across scales |

---
## 2. Normal (Gaussian) Distribution

The bell curve. Most important distribution in statistics because of the **Central Limit Theorem**.

**68-95-99.7 Rule:** 68% of data falls within 1σ, 95% within 2σ, 99.7% within 3σ of the mean.

In [ ]:
## Normal distribution — visualization and properties

import matplotlib.pyplot as plt

np.random.seed(42)
data = np.random.normal(loc=100, scale=15, size=10000)  # Mean=100, Std=15

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with normal curve overlay
axes[0].hist(data, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')
x = np.linspace(40, 160, 200)
axes[0].plot(x, stats.norm.pdf(x, 100, 15), 'r-', linewidth=2, label='Normal PDF')
axes[0].axvline(100, color='green', linestyle='--', label='Mean (μ)')
axes[0].axvline(100-15, color='orange', linestyle=':', label='μ ± 1σ')
axes[0].axvline(100+15, color='orange', linestyle=':')
axes[0].set_title('Normal Distribution (μ=100, σ=15)')
axes[0].legend()

# Q-Q plot — check if data follows normal distribution
stats.probplot(data, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (Normal)')

plt.tight_layout()
plt.show()

# If Q-Q plot points lie on the diagonal line, data is approximately normal.
# Deviations at tails indicate heavy tails (outliers) or skewness.

---
## 3. Other Important Distributions

In [ ]:
## Common distributions side by side

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Binomial — number of successes in n trials
# Example: flip a coin 20 times, count heads
x_binom = np.arange(0, 21)
axes[0,0].bar(x_binom, stats.binom.pmf(x_binom, n=20, p=0.5), color='steelblue')
axes[0,0].set_title('Binomial (n=20, p=0.5)\n# of heads in 20 flips')

# Poisson — number of events in a fixed interval
# Example: customer arrivals per hour (avg 5)
x_pois = np.arange(0, 15)
axes[0,1].bar(x_pois, stats.poisson.pmf(x_pois, mu=5), color='coral')
axes[0,1].set_title('Poisson (λ=5)\nCustomer arrivals per hour')

# Uniform — equal probability everywhere
x_unif = np.linspace(0, 10, 1000)
axes[1,0].plot(x_unif, stats.uniform.pdf(x_unif, loc=2, scale=6), 'g-', linewidth=2)
axes[1,0].fill_between(x_unif, stats.uniform.pdf(x_unif, loc=2, scale=6), alpha=0.3, color='green')
axes[1,0].set_title('Uniform (a=2, b=8)\nRandom number generator')

# Exponential — time between events
# Example: time between customer arrivals
x_exp = np.linspace(0, 5, 200)
axes[1,1].plot(x_exp, stats.expon.pdf(x_exp, scale=1/2), color='purple', linewidth=2)
axes[1,1].fill_between(x_exp, stats.expon.pdf(x_exp, scale=1/2), alpha=0.3, color='purple')
axes[1,1].set_title('Exponential (λ=2)\nTime between arrivals')

plt.tight_layout()
plt.show()

### Distribution Quick Reference

| Distribution | Type | Models | Real-World Example |
|---|---|---|---|
| **Normal** | Continuous | Symmetric bell curve | Heights, IQ scores, measurement errors |
| **Binomial** | Discrete | # successes in n trials | Pass/fail counts, defect rates |
| **Poisson** | Discrete | # events in fixed interval | Calls per hour, website hits |
| **Uniform** | Continuous | Equal probability | Random number generators, dice |
| **Exponential** | Continuous | Time between events | Wait times, component lifetimes |
| **Log-Normal** | Continuous | Right-skewed positive data | Income, stock prices |
| **t-Distribution** | Continuous | Like normal but heavier tails | Small sample inference |

---
## 4. Central Limit Theorem (CLT)

**The most important theorem in statistics:**

> The distribution of **sample means** approaches a normal distribution as sample size increases — *regardless of the population's shape*.

This is why the normal distribution is everywhere and why hypothesis tests work.

In [ ]:
## Central Limit Theorem — visual demonstration

np.random.seed(42)

# Start with a VERY non-normal population (exponential)
population = np.random.exponential(scale=5, size=100000)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Plot 1: Original population (skewed)
axes[0].hist(population, bins=50, density=True, color='lightcoral', edgecolor='white')
axes[0].set_title('Population (Exponential)\nHighly Skewed')

# Plot 2-4: Distribution of sample means for increasing sample sizes
for idx, n in enumerate([5, 30, 100]):
    sample_means = [np.random.choice(population, size=n).mean() for _ in range(5000)]
    axes[idx+1].hist(sample_means, bins=50, density=True, color='steelblue', edgecolor='white')
    axes[idx+1].set_title(f'Sample Means (n={n})\nSkew={stats.skew(sample_means):.2f}')

plt.suptitle('Central Limit Theorem: Sample Means Become Normal', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Even though the population is highly skewed (exponential),
# the distribution of sample means becomes normal by n=30.
# This is the magic of CLT — and why n≥30 is the classic rule of thumb.

> **Key Takeaway:** CLT tells us we can use normal-distribution-based tests (Z-test, t-test) even when the underlying data is not normal — as long as the sample size is large enough (typically n ≥ 30). This is the theoretical foundation of most hypothesis tests.